In [4]:
import numpy as np
from firedrake import *
from firedrake.cython import dmcommon
from petsc4py import PETSc
import math

In [ ]:
# parameters in SI units
t_end = 5.0  # time of simulation [s]
dt = 0.005  # time step [s]
g = 9.8  # gravitational acceleration
# water
Lx = 20.0  # length of the tank [m] in x-direction; needed for computing initial condition
Lz = 10.0  # height of the tank [m]; needed for computing initial condition
rho = 1000.0  # fluid density in kg/m^2 in 2D [water]
# solid parameters
#  - we use a sufficiently soft material to be able to see noticeable structural displacement
rho_B = 7700.0  # structure density in kg/m^2 in 2D
lam = 1e7  # N/m in 2D - first Lame constant
mu = 1e7  # N/m in 2D - second Lame constant

# these numbers must match the ones defined in the mesh file
fluid_id = 1  # fluid subdomain
structure_id = 2  # structure subdomain
bottom_id = 1  # structure bottom
top_id = 6  # fluid surface
interface_id = 9  # fluid-structure interface

L = Lz
T = L / math.sqrt(g * L)
t_end /= T
dt /= T
Lx /= L
Lz /= L
rho_B /= rho
lam /= g * rho * L
mu /= g * rho * L
rho = 1.0  # or equivalently rho /= rho

mesh = Mesh("/home/charlottecai/MSc_Project_new/L_domain.msh")
dim = 2

# Extract submesh
mesh_F = Submesh(mesh, dim, fluid_id,label_name=dmcommon.CELL_SETS_LABEL, name="fluid_mesh")
x_F = SpatialCoordinate(mesh_F)
n_F = FacetNormal(mesh_F)

mesh_S = Submesh(mesh, dim, structure_id, label_name=dmcommon.CELL_SETS_LABEL, name="solid_mesh")
x_S = SpatialCoordinate(mesh_S)
n_S = FacetNormal(mesh_S)

# Function spaces
V_W = FunctionSpace(mesh_F, "CG", 1)
V_B = VectorFunctionSpace(mesh_S, "CG", 1)

mixed_V = V_W * V_B

# Measures
dx_F = Measure("dx", mesh_F)
dx_S = Measure("dx", mesh_S)

# interface measures
ds_F = Measure("ds", mesh_F, intersect_measures=(Measure("ds", mesh_S),))
ds_S = Measure("ds", mesh_S, intersect_measures=(Measure("ds", mesh_F),))

# fluid domain
phi = Function(V_W, name="phi")
phi_f = Function(V_W, name="phi_f")
eta = Function(V_W, name="eta")

trial_W = TrialFunction(V_W)
v_W = TestFunction(V_W)

# Structure domain
X = Function(V_B, name="X")
U = Function(V_B, name="U")

trial_B = TrialFunction(V_B)
v_B = TestFunction(V_B)

# mixed domain
trial_f, trial_s = TrialFunctions(mixed_V)
v_f, v_s = TestFunctions(mixed_V)

tmp_f = Function(V_W)
tmp_s = Function(V_B)
result_mixed = Function(mixed_V)

# Boundary conditions
BC_bottom = DirichletBC(V_B, as_vector([0.0, 0.0]), bottom_id)
BC_bottom_mixed = DirichletBC(mixed_V.sub(1), as_vector([0.0, 0.0]), bottom_id)
BC_phi_f = DirichletBC(mixed_V.sub(0), phi_f, top_id)

# solvers
# 1. equations for phi at the free surface
a_phi_f = trial_W * v_W * ds(top_id)
L_phi_f = (phi_f - dt * eta) * v_W * ds(top_id)
LVP_phi_f = LinearVariationalProblem(a_phi_f, L_phi_f, phi_f)
LVS_phi_f = LinearVariationalSolver(LVP_phi_f)

# 2. equations for  X (beam displacement)
a_X = dot(trial_B, v_B) * dx(structure_id)
L_X = dot((X + dt * U), v_B) * dx(structure_id)

LVP_X = LinearVariationalProblem(a_X, L_X, X, bcs=[BC_bottom])
LVS_X = LinearVariationalSolver(LVP_X)

# 3. equations for phi at the fluid domain, U eta
delX = nabla_grad(X)
delv_B = nabla_grad(v_s)
T_x_dv = lam * div(X) * div(v_s) + mu * (inner(delX, delv_B + transpose(delv_B)))
a_U = rho_B * dot(trial_s, v_s) * dx(structure_id)
L_U = (rho_B * dot(U, v_s) - dt * T_x_dv) * dx(structure_id)
a_phi = dot(grad(trial_f), grad(v_f)) * dx(fluid_id)

a_U += dot(avg(v_s), n_int) * avg(trial_f) * dS 
L_U += dot(avg(v_s), n_int) * avg(phi) * dS
a_phi += -dot(n_int, avg(trial_s)) * avg(v_f) * dS